### Imports and Paths

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

DATA_DIR = Path("../Datasets/Backblaze")

TRAIN_SET = DATA_DIR / "train_set.csv"
VAL_SET = DATA_DIR / "val_set.csv"
VAL_IDS = DATA_DIR / "val_serial_number_id.csv"
VAL_LABELS = DATA_DIR / "val_label.csv"
TEST_SET = DATA_DIR / "test_set.csv"
TEST_IDS = DATA_DIR / "test_serial_number_id.csv"
TEST_LABELS = DATA_DIR / "test_label.csv"

### Helpers

In [4]:
COST = np.array([
    [0, 1, 3],
    [4, 0, 2],
    [15, 5, 0],
], dtype=int)


def last_readout(df: pd.DataFrame) -> pd.DataFrame:
    """Keep the latest SMART snapshot per drive."""
    df = df.sort_values(["serial_number", "date"])
    return df.drop_duplicates("serial_number", keep="last")


def total_cost(y_true, y_pred, cost=COST) -> int:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return int(cost[y_true, y_pred].sum())


def prepare_training_set(path: Path, horizon_days: int = 60):
    """Create labeled daily samples within the desired horizon."""
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values(["serial_number", "date"])
    failure_dates = (
        df.loc[df["failure"] == 1, ["serial_number", "date"]]
          .drop_duplicates("serial_number", keep="last")
          .set_index("serial_number")["date"]
    )
    df = df[df["serial_number"].isin(failure_dates.index)].copy()
    df["failure_date"] = df["serial_number"].map(failure_dates)
    df["days_to_failure"] = (df["failure_date"] - df["date"]).dt.days
    mask = (df["days_to_failure"] >= 0) & (df["days_to_failure"] <= horizon_days)
    df = df.loc[mask].copy()
    df["label"] = np.select(
        [df["days_to_failure"] <= 10, df["days_to_failure"] <= 20],
        [2, 1],
        default=0
    ).astype(int)
    X = df.drop(columns=["date", "serial_number", "failure", "failure_date", "days_to_failure", "label"])
    y = df["label"]
    return X, y


def prepare_eval_split(data_path: Path, id_path: Path, label_path: Path):
    df = pd.read_csv(data_path, parse_dates=["date"])
    df_last = last_readout(df)
    ids = pd.read_csv(id_path)
    labels = pd.read_csv(label_path)
    merged = (df_last
              .merge(ids, on="serial_number", how="inner")
              .merge(labels, on="id", how="inner"))
    X = merged.drop(columns=["date", "serial_number", "id", "label", "failure"])
    y = merged["label"].astype(int)
    return X, y

### Build supervised datasets

In [12]:
X_train, y_train = prepare_training_set(TRAIN_SET, horizon_days=60)
X_val, y_val = prepare_eval_split(VAL_SET, VAL_IDS, VAL_LABELS)
X_test, y_test = prepare_eval_split(TEST_SET, TEST_IDS, TEST_LABELS)

always_nan_cols = [c for c in X_train.columns if X_train[c].isna().all()]
if always_nan_cols:
    X_train = X_train.drop(columns=always_nan_cols)
    X_val = X_val.drop(columns=always_nan_cols, errors="ignore")
    X_test = X_test.drop(columns=always_nan_cols, errors="ignore")
    print(f"Dropped {len(always_nan_cols)} columns with no observed values in training data.")

print(f"Training samples: {len(X_train):,}  |  Features: {X_train.shape[1]}")
print(pd.Series(y_train).value_counts().sort_index().rename("count"))

print(f"Validation samples: {len(X_val)}")
print(pd.Series(y_val).value_counts().sort_index().rename("count"))
print(f"Test samples: {len(X_test)}")
print(pd.Series(y_test).value_counts().sort_index().rename("count"))

FileNotFoundError: [Errno 2] No such file or directory: '../Datasets/Backblaze/test_label.csv'

### Train random forest classifier

In [14]:
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
    n_jobs=-1,
    class_weight="balanced",
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model),
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['capacity_bytes',
                                                   'smart_1_normalized',
                                                   'smart_1_raw',
                                                   'smart_3_normalized',
                                                   'smart_3_raw',
                                                   'smart_4_normalized',
                                                   'smart_4_raw',
                                                   'smart_5_normalized',
                                                   'smart_5_raw',
                                                   'smart_7_normalized',
                                                   'smart_7_raw',
                                                   'smart_9_normalized',
                                                   'smart_9_raw',
                                                   'smart_10_normalized...
                                                   'smart_188_raw',
                                                   'smart_189_normalized',
                                                   'smart_189_raw',
                                                   'smart_190_normalized',
                                                   'smart_190_raw',
                                                   'smart_191_normalized', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['model'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, n_jobs=-1,
                                        random_state=0))])

### Evaluate with cost-sensitive metrics

In [15]:
def evaluate_split(name, X, y):
    y_pred = pipe.predict(X)
    acc = accuracy_score(y, y_pred)
    macro_f1 = f1_score(y, y_pred, average="macro")
    cost = total_cost(y, y_pred, COST)
    print(f"{name} accuracy: {acc:.3f}")
    print(f"{name} macro-F1: {macro_f1:.3f}")
    print(f"{name} total cost: {cost}")
    print("Classification report:")
    print(classification_report(y, y_pred, digits=3, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y, y_pred))
    print()

for split_name, (X_split, y_split) in [
    ("Validation", (X_val, y_val)),
    ("Test", (X_test, y_test)),
]:
    evaluate_split(split_name, X_split, y_split)

Validation accuracy: 0.605
Validation macro-F1: 0.300
Validation total cost: 549
Classification report:
              precision    recall  f1-score   support

           0      0.703     0.854     0.771       130
           1      0.000     0.000     0.000        30
           2      0.125     0.133     0.129        30

    accuracy                          0.605       190
   macro avg      0.276     0.329     0.300       190
weighted avg      0.500     0.605     0.548       190

Confusion matrix (rows=true, cols=pred):
[[111   0  19]
 [ 21   0   9]
 [ 26   0   4]]

Test accuracy: 0.616
Test macro-F1: 0.338
Test total cost: 505
Classification report:
              precision    recall  f1-score   support

           0      0.694     0.838     0.760       130
           1      0.000     0.000     0.000        30
           2      0.242     0.267     0.254        30

    accuracy                          0.616       190
   macro avg      0.312     0.368     0.338       190
weighted avg   